# 01 — DataFrame Basics
SparkSession, DataFrame API, Spark SQL, Window Functions, CTEs.

In [ ]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


## DataFrame API

In [ ]:
schema = StructType([
    StructField('id',         IntegerType()),
    StructField('name',       StringType()),
    StructField('department', StringType()),
    StructField('salary',     DoubleType()),
    StructField('country',    StringType()),
])
employees = spark.createDataFrame([
    (1,'Alice','Engineering',95000.0,'US'),
    (2,'Bob','Marketing',72000.0,'UK'),
    (3,'Carol','Engineering',88000.0,'US'),
    (4,'Dave','HR',65000.0,'DE'),
    (5,'Eve','Engineering',102000.0,'US'),
    (6,'Frank','Marketing',68000.0,'UK'),
    (7,'Grace','HR',71000.0,'DE'),
    (8,'Hank','Engineering',91000.0,'US'),
], schema)
employees.show()
employees.printSchema()


## Aggregations

In [ ]:
employees.groupBy('department').agg(
    F.count('id').alias('headcount'),
    F.avg('salary').alias('avg_salary'),
    F.max('salary').alias('max_salary'),
).orderBy('avg_salary', ascending=False).show()


## Window Functions & CTEs

In [ ]:
employees.createOrReplaceTempView('employees')
spark.sql("""
SELECT name, department, salary,
       RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rank,
       salary - AVG(salary) OVER (PARTITION BY department)       AS diff_from_avg
FROM employees
""").show()


In [ ]:
spark.sql("""
WITH dept_stats AS (
    SELECT department, AVG(salary) AS avg_sal
    FROM employees GROUP BY department
)
SELECT e.name, e.department, e.salary,
       ROUND(e.salary / d.avg_sal * 100, 1) AS pct_of_avg
FROM employees e JOIN dept_stats d USING (department)
ORDER BY pct_of_avg DESC
""").show()
